# ✨ Train Alexia's Rave Clip AI

This notebook teaches a small AI to turn a sentence like **"ROZ invert colors fast"** into a scene for the clip:
which words to show, the effect, the speed, letter style, colors, background, and mini me.

**How it works**
1. **Practice examples.** A generator writes about 40,000 sentences, each with the right scene for it. You can add your own too.
2. **Meaning numbers.** MiniLM, a small language model the app already uses, turns every word into 384 numbers.
   It stays *frozen*, which means we don't change it.
3. **Training.** A small transformer (about 300k numbers to learn) learns to pick every setting, plus which words to put on the clip.
4. **Export.** The trained weights are squeezed to int8, about 0.4 MB, and saved as `scene-ai.bin`.
   The app runs it right on the phone with no internet.

**To run:** turn on the internet (top menu **Settings → Turn on internet**, or right-hand panel **Session options → Internet**), then click **Run All**. The first cell tells you clearly if the internet is still off.
It takes about 10 minutes on Kaggle's free CPU. No GPU is needed.

In [ ]:
# 1. Check the internet, get the tools + MiniLM (the same 23 MB int8 model the app downloads)
import os, sys, subprocess, urllib.request, socket
try:
    socket.create_connection(("huggingface.co", 443), timeout=5).close()
except OSError:
    raise SystemExit(
        "\n🚫 This notebook can't reach the internet yet.\n"
        "Turn it on: top menu  Settings  ->  Turn on internet\n"
        "   (or the right-hand panel: Session options -> Internet -> On.\n"
        "    If the panel is hidden, click the little arrow at the far right edge.)\n"
        "If there's no Internet switch at all, verify your phone number first:\n"
        "   kaggle.com -> your picture -> Settings -> Phone verification.\n"
        "Then click Run All again.")
need = []
for mod, pkg in [("onnxruntime", "onnxruntime"), ("tokenizers", "tokenizers")]:
    try: __import__(mod)
    except ImportError: need.append(pkg)
if need:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + need)
os.makedirs("minilm/onnx", exist_ok=True)
BASE = "https://huggingface.co/Xenova/all-MiniLM-L6-v2/resolve/main/"
for f in ["onnx/model_quantized.onnx", "tokenizer.json"]:
    if not os.path.exists("minilm/" + f):
        urllib.request.urlretrieve(BASE + f, "minilm/" + f)
print("✅ Internet OK. MiniLM ready:", os.path.getsize("minilm/onnx/model_quantized.onnx") // 1_000_000, "MB")

## 2. The code
These cells save the project's Python files next to the notebook (the same files as `ai/scene_ai/` on your computer).

In [ ]:
%%writefile scene_data.py
"""
Makes practice examples for Alexia's scene AI: a sentence someone might type,
and the scene the clip should show for it.

  "ROZ invert colors fast"  ->  fx=invert sp=3 ... t=ROZ
  "pink bubble letters saying PLUR on a starry night with me dancing"
                            ->  fx=still st=outline fg=FF9CC8 bp=stars av=dance1 t=PLUR

Every example also marks WHICH WORDS go on the screen (the AI learns to pick them out).
The answer fields match the clip's scene line exactly (see PROTOCOL.md).
"""
import random, json

# ---------- what the clip can do (same lists as web-app/scene.js) ----------
FIELDS = {
    "fx": ["still", "scroll", "wave", "bounce", "invert", "rainbow", "glitch", "type", "pulse"],
    "sp": ["1", "2", "3"],
    "st": ["plain", "shadow", "outline"],
    "bp": ["solid", "gradient", "checker", "stripes", "dots", "stars", "hearts", "sparkles", "rainbow"],
    "av": ["none", "smile", "teeth", "ooh", "grin", "chill", "dance1", "dance2", "dance3"],
    "hr": ["straight", "curly"],
    "fg": ["FFFFFF", "000000", "1A0033", "3B2B2E", "F7EEDD", "FF9CC8", "FF2FA0", "FF4A55", "FF9A3C",
           "FFE14D", "8FE36B", "7DFFC8", "5AA8FF", "2F3A9E", "A06BFF", "C9B5FF"],
}
for k in ("ac", "bg", "b2"): FIELDS[k] = FIELDS["fg"]
FIELD_ORDER = ["fx", "sp", "st", "bp", "av", "hr", "fg", "ac", "bg", "b2"]
DEFAULT = dict(fx="still", sp="2", st="shadow", bp="solid", av="none", hr="curly",
               fg="FFFFFF", ac="FF2FA0", bg="1A0033", b2="FF9CC8")

# color words -> the 16 colors the app has
COLORS = {
    "FFFFFF": ["white", "snow white", "bright white"],
    "000000": ["black", "jet black", "pitch black"],
    "1A0033": ["dark purple", "midnight", "deep purple", "midnight purple"],
    "3B2B2E": ["dark brown", "chocolate", "espresso", "charcoal"],
    "F7EEDD": ["cream", "beige", "off white", "vanilla"],
    "FF9CC8": ["pink", "baby pink", "light pink", "pastel pink", "bubblegum"],
    "FF2FA0": ["hot pink", "magenta", "neon pink", "fuchsia"],
    "FF4A55": ["red", "cherry red", "strawberry"],
    "FF9A3C": ["orange", "tangerine", "peach"],
    "FFE14D": ["yellow", "gold", "golden", "lemon", "sunshine yellow"],
    "8FE36B": ["green", "lime", "lime green", "neon green", "slime green"],
    "7DFFC8": ["mint", "teal", "aqua", "seafoam", "turquoise"],
    "5AA8FF": ["blue", "baby blue", "sky blue", "light blue"],
    "2F3A9E": ["navy", "dark blue", "royal blue", "indigo"],
    "A06BFF": ["purple", "violet", "grape", "neon purple"],
    "C9B5FF": ["lavender", "lilac", "pastel purple", "periwinkle"],
}
DARK = {"000000", "1A0033", "3B2B2E", "2F3A9E"}

FX_WORDS = {
    "still": ["still", "not moving", "static", "standing still", "no movement", "just sitting there"],
    "scroll": ["scrolling", "scroll across", "sliding across", "moving across the screen", "like a ticker", "scroll"],
    "wave": ["wavy", "waving", "wave", "wiggly", "wiggling", "like a wave"],
    "bounce": ["bouncing", "bouncy", "bounce", "hopping", "jumping letters", "boing"],
    "invert": ["invert colors", "inverting", "flash", "flashing", "strobe", "invert", "flip the colors", "inverted flashing"],
    "rainbow": ["rainbow letters", "rainbow text", "color changing", "changing colors", "rainbow words", "multicolor letters"],
    "glitch": ["glitchy", "glitch", "glitching", "corrupted", "distorted", "broken signal"],
    "type": ["typewriter", "typing", "type it out", "letter by letter", "typed out", "being typed"],
    "pulse": ["pulsing", "pulse", "beating", "throbbing", "heartbeat", "thumping to the bass"],
}
SP_WORDS = {
    "1": ["slow", "slowly", "super slow", "chill speed", "gently", "lazy", "nice and slow"],
    "2": ["medium speed", "normal speed", "regular speed"],
    "3": ["fast", "quick", "quickly", "super fast", "really fast", "hyper", "crazy fast", "speedy"],
}
ST_WORDS = {
    "plain": ["plain letters", "simple font", "no shadow", "plain text", "flat letters"],
    "shadow": ["with a shadow", "drop shadow", "shadowed letters", "with shadows", "3d letters"],
    "outline": ["bubble letters", "outlined", "outline", "bubble font", "outlined letters", "puffy letters"],
}
BP_WORDS = {
    "solid": ["plain background", "solid background", "simple background", "solid color background"],
    "gradient": ["gradient", "fade", "ombre", "gradient background", "fading background"],
    "checker": ["checkered", "checkerboard", "checks", "checker pattern", "checkered background"],
    "stripes": ["stripes", "striped", "stripey background", "striped background"],
    "dots": ["polka dots", "dots", "dotted", "spots", "polka dot background"],
    "stars": ["stars", "starry", "night sky", "starry sky", "starry background"],
    "hearts": ["hearts", "hearts everywhere", "floating hearts", "lots of hearts", "heart background"],
    "sparkles": ["sparkles", "glitter", "sparkly", "twinkles", "sparkly background", "glittery"],
    "rainbow": ["rainbow background", "rainbow bg", "rainbow behind it", "rainbow sky"],
}
AV_WORDS = {
    "smile": ["me smiling", "mini me smiling", "my avatar smiling", "little me with a smile", "me with a cute smile"],
    "teeth": ["me with bunny teeth", "mini me showing my teeth", "me with my teeth out", "my avatar with bunny teeth"],
    "ooh": ["me making an ooh face", "mini me going ooh", "me blowing a kiss", "my avatar with a kissy face", "me surprised"],
    "grin": ["me grinning", "mini me with a big smile", "me super happy", "my avatar grinning", "me laughing"],
    "chill": ["me chilling", "mini me looking chill", "me with a chill face", "my avatar just vibing", "me unbothered"],
    "dance1": ["me dancing", "mini me dancing", "my avatar dancing", "me doing dance 1", "me doing the first dance",
               "little me dancing", "me grooving"],
    "dance2": ["me doing dance 2", "mini me doing the second dance", "me doing dance two", "my avatar doing dance 2",
               "me shuffling"],
    "dance3": ["me doing dance 3", "mini me doing the third dance", "me doing dance three", "my avatar doing dance 3",
               "me jumping around"],
}
HR_WORDS = {
    "straight": ["straight hair", "my straight hair", "sleek hair", "hair down straight"],
    "curly": ["curly hair", "my curls", "curly", "my curly hair"],
}
AV_PREFIX = ["with", "and", "plus", "", "featuring"]

# vibes set a whole look (colors + effect + background) unless you say otherwise
VIBES = [
    (["cute", "kawaii", "sweet", "girly", "soft"], dict(fg="FFFFFF", ac="FF2FA0", bg="FF9CC8", b2="FFFFFF", bp="hearts", st="outline", fx="bounce")),
    (["spooky", "creepy", "halloween", "goth", "dark"], dict(fg="FF9A3C", ac="000000", bg="000000", b2="A06BFF", bp="stars", st="shadow", fx="glitch")),
    (["techno", "underground", "warehouse", "industrial"], dict(fg="7DFFC8", ac="FF2FA0", bg="000000", b2="2F3A9E", bp="stripes", st="plain", fx="glitch")),
    (["rave", "neon", "edm", "festival", "party"], dict(fg="8FE36B", ac="FF2FA0", bg="000000", b2="FF2FA0", bp="checker", st="shadow", fx="rainbow")),
    (["ocean", "beach", "underwater", "sea"], dict(fg="FFFFFF", ac="2F3A9E", bg="5AA8FF", b2="7DFFC8", bp="dots", st="shadow", fx="wave")),
    (["sunset", "summer", "tropical", "warm"], dict(fg="FFE14D", ac="FF2FA0", bg="FF9A3C", b2="FF2FA0", bp="gradient", st="shadow", fx="wave")),
    (["galaxy", "space", "cosmic", "dreamy"], dict(fg="C9B5FF", ac="A06BFF", bg="1A0033", b2="FFFFFF", bp="stars", st="shadow", fx="pulse")),
    (["romantic", "lovey", "valentine", "in love"], dict(fg="FFFFFF", ac="FF4A55", bg="FF2FA0", b2="FF9CC8", bp="hearts", st="outline", fx="pulse")),
    (["angel", "heavenly", "pure", "clean"], dict(fg="FFFFFF", ac="C9B5FF", bg="5AA8FF", b2="FFFFFF", bp="sparkles", st="outline", fx="still")),
    (["cozy", "autumn", "fall", "coffee"], dict(fg="F7EEDD", ac="3B2B2E", bg="3B2B2E", b2="FF9A3C", bp="dots", st="shadow", fx="still")),
    (["icy", "winter", "frozen", "snowy"], dict(fg="FFFFFF", ac="5AA8FF", bg="2F3A9E", b2="FFFFFF", bp="sparkles", st="outline", fx="still")),
    (["cyber", "y2k", "futuristic", "hacker"], dict(fg="7DFFC8", ac="A06BFF", bg="000000", b2="7DFFC8", bp="checker", st="plain", fx="type")),
    (["hype", "crazy", "wild", "chaotic"], dict(fg="FFE14D", ac="FF2FA0", bg="000000", b2="FFE14D", bp="rainbow", st="shadow", fx="invert", sp="3")),
    (["chill", "lowkey", "mellow", "calm"], dict(fg="C9B5FF", ac="1A0033", bg="1A0033", b2="A06BFF", bp="gradient", st="shadow", fx="wave", sp="1")),
]
VIBE_FORMS = ["{v} vibes", "make it {v}", "{v} vibe", "something {v}", "{v} style", "very {v}", "{v}"]

# words people put on their clip
TEXTS = ["ROZ", "PLUR", "I LOVE ROZ", "HI", "HI!", "XOXO", "RAVE", "TECHNO", "BASS", "DROP IT", "KANDI KID",
         "PEACE", "LOVE", "UNITY", "RESPECT", "LETS GO", "NO SLEEP", "HUG ME", "FREE HUGS", "TRADE KANDI",
         "BASS FACE", "HOUSE MUSIC", "DNB", "DUBSTEP", "TRANCE", "EDC", "PLUR 4 LIFE", "ALEXIA", "BESTIE",
         "WHERE MY FRIENDS", "FIND ME", "LOST", "SOBER RAVER", "HYDRATE", "BPM", "WUBS", "HEADBANG", "SLAY",
         "OMG", "LOL", "YAY", "WOW", "BOOM", "VIBES", "GOOD VIBES", "STAY WEIRD", "BE KIND", "2026", "ROZ ROZ",
         "SKRILLEX", "ODESZA", "ILLENIUM", "ZEDD", "DEADMAU5", "FRED AGAIN", "PEGGY GOU", "FISHER", "JOHN SUMMIT",
         "MAU P", "SARA LANDRY", "INDIRA", "CHRIS LAKE", "ALISON WONDER", "KAYTRANADA", "GRIZ", "SUBTRONICS",
         "LEVEL UP", "SVDDEN DEATH", "EXCISION", "SEVEN LIONS", "GORGON CITY", "DOM DOLLA", "TIESTO", "MARSHMELLO"]
SYL = ["ka", "zo", "ri", "ma", "lu", "ne", "vo", "ta", "shi", "ro", "xa", "bel", "ny", "dee", "jo", "ki", "ra", "sol"]

def random_text(rng):
    r = rng.random()
    if r < 0.55: t = rng.choice(TEXTS)
    elif r < 0.8:   # made-up names, so it learns ANY word can be shown
        t = "".join(rng.choice(SYL) for _ in range(rng.randint(1, 3)))
        if rng.random() < 0.3: t += " " + "".join(rng.choice(SYL) for _ in range(rng.randint(1, 2)))
    elif r < 0.9: t = str(rng.choice([2026, 4, 69, 420, 777, 100, 24, 7]))
    else: t = rng.choice(TEXTS) + " " + rng.choice(["4EVER", "!!", "<3", "XO", "2026"])
    case = rng.random()
    if case < 0.6: t = t.upper()
    elif case < 0.85: t = t.lower()
    else: t = t.title()
    return t[:24].strip()

def text_phrase(rng, t):
    """Returns (words, which ones are the text)."""
    tw = t.split()
    form = rng.random()
    def mk(before, quote=False, after=()):
        words = list(before)
        qw = list(tw)
        if quote:
            q = rng.choice(['"', '"', "'"])
            qw[0] = q + qw[0]; qw[-1] = qw[-1] + q
        tags = [0] * len(words) + [1] * len(qw)
        words += qw
        words += list(after); tags += [0] * len(after)
        return words, tags
    if form < 0.2: return mk([], quote=rng.random() < 0.3)                                  # ROZ ...
    if form < 0.35: return mk(rng.choice([["saying"], ["that", "says"], ["that", "said"]]), quote=rng.random() < 0.5)
    if form < 0.5: return mk(rng.choice([["the", "word"], ["the", "words"], ["write"], ["text"], ["show"], ["put"]]), quote=rng.random() < 0.5)
    if form < 0.65: return mk([], quote=True)
    if form < 0.75: return mk(rng.choice([["my", "clip", "says"], ["it", "says"], ["spell", "out"]]), quote=rng.random() < 0.4)
    if form < 0.85: return mk([], quote=rng.random() < 0.5, after=rng.choice([["in", "big", "letters"], ["on", "my", "clip"], ["on", "the", "screen"]]))
    return mk(rng.choice([["with", "the", "text"], ["with", "the", "words"], ["reading"]]), quote=rng.random() < 0.6)

def contrast(fg, bg, rng):
    if (fg in DARK) == (bg in DARK) or fg == bg:
        return rng.choice(["FFFFFF", "FFE14D", "FF9CC8", "7DFFC8"]) if bg in DARK else rng.choice(["000000", "1A0033", "2F3A9E"])
    return fg

def cword(rng, hexv): return rng.choice(COLORS[hexv])

def make_example(rng):
    scene = dict(DEFAULT)
    k = rng.choice([0.35, 0.6, 1.0])   # short, medium or long sentences
    phrases = []            # list of (words, tags)
    said = set()
    # 1) vibe (sets defaults that explicit words can override)
    if rng.random() < 0.35 * k:
        names, look = rng.choice(VIBES)
        scene.update(look)
        phrases.append((rng.choice(VIBE_FORMS).format(v=rng.choice(names)).split(), None))
    # 2) text
    t = random_text(rng) if rng.random() < 0.85 else ""
    tp = text_phrase(rng, t) if t else None
    # 3) effect, speed, style
    if rng.random() < 0.7 * k:
        fx = rng.choice(FIELDS["fx"]); scene["fx"] = fx; said.add("fx")
        phrases.append((rng.choice(FX_WORDS[fx]).split(), None))
    if rng.random() < 0.45 * k:
        sp = rng.choice(["1", "3", "3", "1", "2"]); scene["sp"] = sp; said.add("sp")
        phrases.append((rng.choice(SP_WORDS[sp]).split(), None))
    if rng.random() < 0.35 * k:
        st = rng.choice(FIELDS["st"]); scene["st"] = st; said.add("st")
        phrases.append((rng.choice(ST_WORDS[st]).split(), None))
    # 4) colors
    if rng.random() < 0.55 * k:
        fg = rng.choice(FIELDS["fg"]); scene["fg"] = fg; said.add("fg")
        phrases.append((rng.choice(["{c} letters", "{c} text", "in {c}", "{c} words", "make the letters {c}", "{c} font"]).format(c=cword(rng, fg)).split(), None))
    if rng.random() < 0.25 * k:
        ac = rng.choice(FIELDS["ac"]); scene["ac"] = ac; said.add("ac")
        phrases.append((rng.choice(["{c} shadow", "{c} outline", "with a {c} outline", "{c} bubble", "{c} glow"]).format(c=cword(rng, ac)).split(), None))
    # 5) background
    bp = None
    if rng.random() < 0.55 * k:
        bp = rng.choice(FIELDS["bp"]); scene["bp"] = bp; said.add("bp")
    if rng.random() < 0.5 * k:
        bg = rng.choice(FIELDS["bg"]); scene["bg"] = bg; said.add("bg")
        if bp in ("gradient",) and rng.random() < 0.6:
            b2 = rng.choice(FIELDS["b2"]); scene["b2"] = b2; said.add("b2")
            phrases.append((rng.choice(["{a} to {b} gradient", "{a} and {b} ombre", "fade from {a} to {b}"]).format(a=cword(rng, bg), b=cword(rng, b2)).split(), None))
        elif bp and bp not in ("solid", "rainbow", "gradient") and rng.random() < 0.5:
            b2 = rng.choice(FIELDS["b2"]); scene["b2"] = b2; said.add("b2")
            pw = rng.choice(BP_WORDS[bp]).replace(" background", "")
            phrases.append((rng.choice(["{p} in {c2} on {c}", "{c2} {p} on a {c} background", "{c} background with {c2} {p}"]).format(p=pw, c=cword(rng, bg), c2=cword(rng, b2)).split(), None))
        elif bp == "gradient" and rng.random() < 0.5:
            phrases.append((f"{cword(rng, bg)} {rng.choice(['gradient', 'ombre', 'fade', 'gradient background'])}".split(), None))
        else:
            phrases.append((rng.choice(["on a {c} background", "{c} background", "on {c}", "{c} bg", "with a {c} background"]).format(c=cword(rng, bg)).split(), None))
            if bp: phrases.append((rng.choice(BP_WORDS[bp]).split(), None))
    elif bp:
        if bp not in ("solid", "rainbow", "gradient") and rng.random() < 0.35:
            b2 = rng.choice(FIELDS["b2"]); scene["b2"] = b2; said.add("b2")
            pw = rng.choice(BP_WORDS[bp]).replace(" background", "")
            phrases.append((f"{cword(rng, b2)} {pw}".split(), None))
        else:
            phrases.append((rng.choice(BP_WORDS[bp]).split(), None))
    # 6) mini me
    if rng.random() < 0.4 * k:
        av = rng.choice(FIELDS["av"][1:]); scene["av"] = av
        words = rng.choice(AV_WORDS[av]).split()
        pre = rng.choice(AV_PREFIX)
        if pre: words = [pre] + words
        if rng.random() < 0.5:
            hr = rng.choice(FIELDS["hr"]); scene["hr"] = hr
            words += ["with"] + rng.choice(HR_WORDS[hr]).split()
        phrases.append((words, None))
    # make sure the words stay readable
    if "fg" not in said: scene["fg"] = contrast(scene["fg"], scene["bg"], rng)
    # put it together in a random order (text often first, like "ROZ invert colors fast")
    rng.shuffle(phrases)
    if tp:
        pos = 0 if rng.random() < 0.45 else rng.randint(0, len(phrases))
        phrases.insert(pos, tp)
    if not phrases:
        return None
    words, tags = [], []
    for i, (w, tg) in enumerate(phrases):
        if i and rng.random() < 0.25:
            j = rng.choice([",", "and", "with", "+"])
            if j != w[0]: words.append(j); tags.append(0)
        words += w; tags += tg if tg else [0] * len(w)
    if rng.random() < 0.15: words = rng.choice([["make"], ["show"], ["i", "want"], ["can", "you", "do"], ["pls"]]) + words; tags = [0] * (len(words) - len(tags)) + tags
    if rng.random() < 0.1: words += [rng.choice(["pls", "please", "!!", "lol", "thx"])]; tags += [0]
    # glue commas onto the word before them (people type "ROZ, fast")
    out_w, out_t = [], []
    for w, tg in zip(words, tags):
        if w == "," and out_w: out_w[-1] += ","; continue
        out_w.append(w); out_t.append(tg)
    scene["t"] = t
    return dict(words=out_w, tags=out_t, scene=scene)

def make_dataset(n, seed=0):
    rng = random.Random(seed); out = []
    while len(out) < n:
        ex = make_example(rng)
        if ex and len(ex["words"]) <= 40: out.append(ex)
    return out

def scene_line(s):
    return ";".join(f"{k}={s[k]}" for k in FIELD_ORDER) + f";t={s['t']}"

def clean_word(w):
    """A word as it should appear on the clip: no quotes, no trailing comma."""
    return w.strip("\"“”‘’").rstrip(",").strip("'") or w

if __name__ == "__main__":
    for ex in make_dataset(12, seed=1):
        print(" ".join(ex["words"]))
        print("   text:", " ".join(w for w, t in zip(ex["words"], ex["tags"]) if t), "|", scene_line(ex["scene"]))

In [ ]:
%%writefile scene_model.py
"""
Alexia's scene AI.

  sentence --(MiniLM, frozen, the same 23 MB model the app already downloads)--> one vector per word
           --(this small model, trained by us)--> the scene: effect, speed, letters, background,
                                                   colors, mini-me, hair + which words to show

The small model: a 2-layer transformer over the words, then
  - a "show this word?" yes/no for every word (picks the text out of the sentence)
  - one attention head per setting (each setting learns which words to listen to)
It is written by hand (no nn.Transformer) so the app can run the exact same math in plain JavaScript.
"""
import math, json, struct
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from scene_data import FIELDS, FIELD_ORDER

D_IN, D, HEADS, FF, LAYERS, MAX_WORDS = 384, 128, 4, 256, 2, 48

class Layer(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(D); self.qkv = nn.Linear(D, 3 * D); self.out = nn.Linear(D, D)
        self.ln2 = nn.LayerNorm(D); self.ff1 = nn.Linear(D, FF); self.ff2 = nn.Linear(FF, D)
    def forward(self, x, mask):            # x [B,W,D], mask [B,W] (1 = real word)
        B, W, _ = x.shape; h = D // HEADS
        q, k, v = self.qkv(self.ln1(x)).view(B, W, 3, HEADS, h).permute(2, 0, 3, 1, 4)
        att = (q @ k.transpose(-1, -2)) / math.sqrt(h)
        att = att.masked_fill(mask[:, None, None, :] == 0, -1e4).softmax(-1)
        x = x + self.out((att @ v).transpose(1, 2).reshape(B, W, D))
        return x + self.ff2(F.gelu(self.ff1(self.ln2(x)), approximate="tanh"))

class SceneAI(nn.Module):
    def __init__(self):
        super().__init__()
        self.inp = nn.Linear(D_IN, D); self.pos = nn.Parameter(torch.zeros(MAX_WORDS, D))
        self.layers = nn.ModuleList(Layer() for _ in range(LAYERS)); self.ln = nn.LayerNorm(D)
        self.tag = nn.Linear(D, 1)
        self.query = nn.Parameter(torch.randn(len(FIELD_ORDER), D) * 0.02)
        self.heads = nn.ModuleList(nn.Linear(D, len(FIELDS[f])) for f in FIELD_ORDER)
        self.drop = nn.Dropout(0.1)
    def forward(self, words, mask):        # words [B,W,384] -> (tag logits [B,W], [logits per setting])
        W = words.shape[1]
        x = self.drop(self.inp(words) + self.pos[:W])
        for L in self.layers: x = L(x, mask)
        x = self.ln(x)
        tag = self.tag(x).squeeze(-1)
        att = (x @ self.query.T) / math.sqrt(D)                       # [B,W,fields]
        att = att.masked_fill(mask[:, :, None] == 0, -1e4).softmax(1)
        pooled = att.transpose(1, 2) @ x                              # [B,fields,D]
        return tag, [h(self.drop(pooled[:, i])) for i, h in enumerate(self.heads)]

# ---------- MiniLM word vectors (same steps as the app) ----------
class WordVectors:
    """Runs MiniLM (ONNX, int8) and keeps the vector of each word's first piece."""
    def __init__(self, onnx_path, tokenizer_path):
        import onnxruntime as ort
        from tokenizers import Tokenizer
        self.sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
        self.tok = Tokenizer.from_file(tokenizer_path)
        self.tok.no_padding(); self.tok.no_truncation()   # one word at a time, exactly like the app
        self.cache = {}
    def pieces(self, w):
        if w not in self.cache:
            ids = self.tok.encode(w, add_special_tokens=False).ids
            self.cache[w] = ids or [100]                 # [UNK]
        return self.cache[w]
    def __call__(self, sentences, batch=64):
        """sentences: list of word lists -> list of float32 arrays [n_words, 384]"""
        out = []
        for b in range(0, len(sentences), batch):
            chunk = sentences[b:b + batch]; rows, firsts = [], []
            for words in chunk:
                ids, first = [101], []
                for w in words[:MAX_WORDS]:
                    p = self.pieces(w)
                    if len(ids) + len(p) > 126: p = p[:max(1, 126 - len(ids))]
                    first.append(len(ids)); ids += p
                ids.append(102); rows.append(ids); firsts.append(first)
            T = max(map(len, rows))
            ids = np.zeros((len(rows), T), np.int64); am = np.zeros_like(ids)
            for i, r in enumerate(rows): ids[i, :len(r)] = r; am[i, :len(r)] = 1
            feeds = {"input_ids": ids, "attention_mask": am}
            if "token_type_ids" in [i.name for i in self.sess.get_inputs()]: feeds["token_type_ids"] = np.zeros_like(ids)
            hid = self.sess.run(None, feeds)[0]
            for i, f in enumerate(firsts): out.append(hid[i, f].astype(np.float32))
        return out

# ---------- saving for the app: int8 weights + a small JSON header in one file ----------
def export(model, path, info=None):
    """scene-ai.bin = 'SCAI' | header length (uint32) | JSON header | int8 weights.
    Each weight matrix is stored as int8 with one scale per row (value = int8 * scale)."""
    model.eval(); tensors, blobs, off = [], [], 0
    for name, t in model.state_dict().items():
        a = t.detach().float().cpu().numpy()
        a2 = a.reshape(a.shape[0], -1) if a.ndim > 1 else a.reshape(1, -1)
        scale = np.abs(a2).max(1) / 127.0; scale[scale == 0] = 1e-8
        q = np.clip(np.round(a2 / scale[:, None]), -127, 127).astype(np.int8)
        sc = scale.astype(np.float32).tobytes(); qb = q.tobytes()
        tensors.append(dict(name=name, shape=list(a.shape), rows=int(a2.shape[0]), scale_at=off, data_at=off + len(sc)))
        blob = sc + qb; blob += b"\0" * (-len(blob) % 4); blobs.append(blob); off += len(blob)
    head = dict(version=1, d_in=D_IN, d=D, heads=HEADS, ff=FF, layers=LAYERS, max_words=MAX_WORDS,
                fields=FIELD_ORDER, choices={f: FIELDS[f] for f in FIELD_ORDER}, tensors=tensors, info=info or {})
    hb = json.dumps(head).encode(); hb += b" " * (-(len(hb) + 8) % 4)
    with open(path, "wb") as f:
        f.write(b"SCAI"); f.write(struct.pack("<I", len(hb))); f.write(hb); f.write(b"".join(blobs))
    return 8 + len(hb) + off

def load_int8(model, path):
    """Loads scene-ai.bin back into a model (to check the int8 version still works)."""
    raw = open(path, "rb").read(); n = struct.unpack("<I", raw[4:8])[0]; head = json.loads(raw[8:8 + n]); base = 8 + n
    sd = {}
    for t in head["tensors"]:
        rows = t["rows"]; size = int(np.prod(t["shape"])); cols = size // rows
        sc = np.frombuffer(raw, np.float32, rows, base + t["scale_at"])
        q = np.frombuffer(raw, np.int8, size, base + t["data_at"]).reshape(rows, cols).astype(np.float32)
        sd[t["name"]] = torch.tensor((q * sc[:, None]).reshape(t["shape"]))
    model.load_state_dict(sd); return model

In [ ]:
%%writefile train.py
"""
Trains Alexia's scene AI. The Kaggle notebook runs these same steps; this file is for running on a laptop.
  python train.py --minilm <folder with onnx/model_quantized.onnx and tokenizer.json> [--n 40000] [--epochs 12]
"""
import argparse, json, os, random, time
import numpy as np, torch, torch.nn.functional as F
from scene_data import make_dataset, FIELDS, FIELD_ORDER, clean_word, scene_line
from scene_model import SceneAI, WordVectors, export, load_int8, MAX_WORDS

HERE = os.path.dirname(os.path.abspath(__file__))

def encode(examples, wv):
    vecs = wv([e["words"] for e in examples])
    ys = [[FIELDS[f].index(e["scene"][f]) for f in FIELD_ORDER] for e in examples]
    tags = [e["tags"][:MAX_WORDS] for e in examples]
    avon = [e["scene"]["av"] != "none" for e in examples]
    return vecs, ys, tags, avon

def batches(data, bs, shuffle):
    vecs, ys, tags, avon = data; idx = list(range(len(vecs)))
    if shuffle: random.shuffle(idx)
    for b in range(0, len(idx), bs):
        ii = idx[b:b + bs]; W = max(len(vecs[i]) for i in ii)
        x = np.zeros((len(ii), W, 384), np.float32); m = np.zeros((len(ii), W), np.float32); t = np.zeros((len(ii), W), np.float32)
        for j, i in enumerate(ii):
            n = len(vecs[i]); x[j, :n] = vecs[i]; m[j, :n] = 1; t[j, :n] = tags[i]
        yield (torch.tensor(x), torch.tensor(m), torch.tensor(t),
               torch.tensor([ys[i] for i in ii]), torch.tensor([avon[i] for i in ii], dtype=torch.float32))

HR = FIELD_ORDER.index("hr")
def loss_fn(model, x, m, t, y, avon):
    tag, outs = model(x, m)
    loss = (F.binary_cross_entropy_with_logits(tag, t, reduction="none") * m).sum() / m.sum()
    for i, o in enumerate(outs):
        l = F.cross_entropy(o, y[:, i], reduction="none")
        if i == HR: l = l * avon            # hair only matters when mini-me is on
        loss = loss + l.mean()
    return loss, tag, outs

def evaluate(model, data):
    model.eval(); right = np.zeros(len(FIELD_ORDER)); n = 0; text_ok = 0
    with torch.no_grad():
        for x, m, t, y, avon in batches(data, 256, False):
            tag, outs = model(x, m)
            for i, o in enumerate(outs): right[i] += (o.argmax(1) == y[:, i]).sum().item()
            text_ok += (((tag > 0).float() == t) | (m == 0)).all(1).sum().item(); n += len(y)
    model.train()
    return {f: right[i] / n for i, f in enumerate(FIELD_ORDER)} | {"text": text_ok / n}

def train(train_data, val_data, epochs=12, lr=2e-3, log=print):
    torch.manual_seed(0); random.seed(0)
    model = SceneAI(); opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps = epochs * ((len(train_data[0]) + 127) // 128)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps)
    for ep in range(epochs):
        t0 = time.time(); tot = 0; k = 0
        for x, m, t, y, avon in batches(train_data, 128, True):
            # a little noise, so it copes with the phone's slightly different numbers
            x = x + torch.randn_like(x) * 0.02
            loss, _, _ = loss_fn(model, x, m, t, y, avon)
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
            tot += loss.item(); k += 1
        acc = evaluate(model, val_data)
        log(f"round {ep + 1}/{epochs}  loss {tot / k:.3f}  text {acc['text']:.1%}  effect {acc['fx']:.1%}  "
            f"background {acc['bp']:.1%}  mini-me {acc['av']:.1%}  letters color {acc['fg']:.1%}  ({time.time() - t0:.0f}s)")
    return model

def predict(model, wv, sentence):
    words = sentence.split()[:MAX_WORDS]
    if not words: return None
    x = torch.tensor(wv([words])[0])[None]; m = torch.ones(1, len(words))
    model.eval()
    with torch.no_grad(): tag, outs = model(x, m)
    scene = {f: FIELDS[f][o[0].argmax().item()] for f, o in zip(FIELD_ORDER, outs)}
    scene["t"] = " ".join(clean_word(w) for w, k in zip(words, tag[0]) if k > 0)[:24]
    return scene

def check_tests(model, wv, tests, log=print):
    """tests: [{prompt, expect: {field: value}}] written by hand, phrased differently from the practice examples."""
    ok = tot = 0; whole = 0
    for tcase in tests:
        got = predict(model, wv, tcase["prompt"]); bad = []
        for f, v in tcase["expect"].items():
            tot += 1
            if str(got[f]).lower() == str(v).lower(): ok += 1
            else: bad.append(f"{f}: wanted {v}, got {got[f]}")
        whole += not bad
        log(("✅ " if not bad else "❌ ") + tcase["prompt"] + ("" if not bad else "   (" + "; ".join(bad) + ")"))
    log(f"\nHand-written tests: {whole}/{len(tests)} sentences fully right, {ok}/{tot} settings right ({ok / tot:.0%})")
    return ok / tot

if __name__ == "__main__":
    ap = argparse.ArgumentParser(); ap.add_argument("--minilm", required=True); ap.add_argument("--n", type=int, default=40000)
    ap.add_argument("--epochs", type=int, default=12); ap.add_argument("--out", default="scene-ai.bin")
    a = ap.parse_args()
    wv = WordVectors(os.path.join(a.minilm, "onnx", "model_quantized.onnx"), os.path.join(a.minilm, "tokenizer.json"))
    t0 = time.time()
    tr = encode(make_dataset(a.n, seed=1), wv); va = encode(make_dataset(2000, seed=2), wv)
    print(f"word vectors ready in {time.time() - t0:.0f}s")
    model = train(tr, va, a.epochs)
    tests = json.load(open(os.path.join(HERE, "test_prompts.json")))
    check_tests(model, wv, tests)
    size = export(model, a.out, info=dict(examples=a.n, epochs=a.epochs))
    print(f"saved {a.out} ({size / 1024:.0f} KB)")
    m2 = load_int8(SceneAI(), a.out); print("int8 version:"); check_tests(m2, wv, tests, log=lambda s: None if s[:1] in "✅❌" else print(s))

def example_from_line(prompt, line):
    """Turns one of YOUR examples ("ROZ invert colors fast", "fx=invert;sp=3;t=ROZ") into practice data.
    Settings you leave out get the normal defaults. The words in t= are marked as the words to show."""
    from scene_data import DEFAULT
    scene = dict(DEFAULT); scene["t"] = ""
    ti = line.find("t=")
    if ti >= 0 and (ti == 0 or line[ti - 1] == ";"): scene["t"] = line[ti + 2:]; line = line[:ti]
    for part in line.split(";"):
        if "=" in part:
            k, v = part.split("=", 1); k = k.strip(); v = v.strip()
            if k in FIELDS:
                v = v.upper() if k in ("fg", "ac", "bg", "b2") else v
                if v not in FIELDS[k]: raise ValueError(f"{k}={v} isn't one of {FIELDS[k]}")
                scene[k] = v
    words = prompt.split(); want = [w.lower() for w in scene["t"].split()]; tags = [0] * len(words)
    for i in range(len(words)):
        if want and [clean_word(w).lower() for w in words[i:i + len(want)]] == want:
            tags[i:i + len(want)] = [1] * len(want); break
    else:
        if want: raise ValueError(f'Couldn\'t find the words "{scene["t"]}" in "{prompt}"')
    return dict(words=words, tags=tags, scene=scene)

In [ ]:
%%writefile test_prompts.json
[
 {"prompt": "ROZ invert colors fast", "expect": {"t": "ROZ", "fx": "invert", "sp": "3"}},
 {"prompt": "the word PLUR in pink bubble letters", "expect": {"t": "PLUR", "fg": "FF9CC8", "st": "outline"}},
 {"prompt": "I LOVE ROZ with hearts", "expect": {"t": "I LOVE ROZ", "bp": "hearts"}},
 {"prompt": "\"TECHNO\" glitchy on black", "expect": {"t": "TECHNO", "fx": "glitch", "bg": "000000"}},
 {"prompt": "me dancing and HI!", "expect": {"t": "HI!", "av": "dance1"}},
 {"prompt": "rainbow text saying rave", "expect": {"t": "rave", "fx": "rainbow"}},
 {"prompt": "slow wavy FISHER", "expect": {"t": "FISHER", "fx": "wave", "sp": "1"}},
 {"prompt": "sparkly purple background with XOXO", "expect": {"t": "XOXO", "bp": "sparkles", "bg": "A06BFF"}},
 {"prompt": "mini me doing dance 2 with straight hair", "expect": {"t": "", "av": "dance2", "hr": "straight"}},
 {"prompt": "starry night sky with me smiling", "expect": {"t": "", "bp": "stars", "av": "smile"}},
 {"prompt": "write BASS in yellow, pulsing", "expect": {"t": "BASS", "fg": "FFE14D", "fx": "pulse"}},
 {"prompt": "typewriter HELLO", "expect": {"t": "HELLO", "fx": "type"}},
 {"prompt": "KANDI KID bouncing on a mint background", "expect": {"t": "KANDI KID", "fx": "bounce", "bg": "7DFFC8"}},
 {"prompt": "spooky vibes BOO", "expect": {"t": "BOO", "fx": "glitch"}},
 {"prompt": "show my name ALEXIA with a hot pink outline", "expect": {"t": "ALEXIA", "ac": "FF2FA0"}},
 {"prompt": "checkered background blue and white", "expect": {"t": "", "bp": "checker"}},
 {"prompt": "scrolling DOM DOLLA super fast", "expect": {"t": "DOM DOLLA", "fx": "scroll", "sp": "3"}},
 {"prompt": "my avatar with bunny teeth saying OMG", "expect": {"t": "OMG", "av": "teeth"}},
 {"prompt": "polka dots and me doing dance 3", "expect": {"t": "", "bp": "dots", "av": "dance3"}},
 {"prompt": "red letters ZEDD on a navy background", "expect": {"t": "ZEDD", "fg": "FF4A55", "bg": "2F3A9E"}},
 {"prompt": "FRED AGAIN with stripes", "expect": {"t": "FRED AGAIN", "bp": "stripes"}},
 {"prompt": "flashing HYDRATE slowly", "expect": {"t": "HYDRATE", "fx": "invert", "sp": "1"}},
 {"prompt": "lavender gradient", "expect": {"t": "", "bp": "gradient", "bg": "C9B5FF"}},
 {"prompt": "\"free hugs\" plain letters", "expect": {"t": "free hugs", "st": "plain"}},
 {"prompt": "me blowing a kiss with curly hair", "expect": {"t": "", "av": "ooh", "hr": "curly"}},
 {"prompt": "cute PLUR", "expect": {"t": "PLUR", "bp": "hearts"}},
 {"prompt": "GRIZ rainbow background", "expect": {"t": "GRIZ", "bp": "rainbow"}},
 {"prompt": "drop shadow letters saying TRANCE in lilac", "expect": {"t": "TRANCE", "st": "shadow", "fg": "C9B5FF"}},
 {"prompt": "WUBS glitch fast green", "expect": {"t": "WUBS", "fx": "glitch", "sp": "3", "fg": "8FE36B"}},
 {"prompt": "sara landry and me chilling", "expect": {"t": "sara landry", "av": "chill"}}
]

## 3. Your own examples (optional, but this is what makes it *yours*)
Add sentences the way YOU would type them, with the scene you want. Each one counts 30 times in training.
Leave out any setting you don't care about. Use the words exactly as they appear in the app:
- `fx` effect: still, scroll, wave, bounce, invert, rainbow, glitch, type, pulse
- `sp` speed: 1 slow, 2 medium, 3 fast
- `st` letters: plain, shadow, outline
- `bp` background: solid, gradient, checker, stripes, dots, stars, hearts, sparkles, rainbow
- `av` mini me: none, smile, teeth, ooh, grin, chill, dance1, dance2, dance3, and `hr` hair: straight, curly
- colors `fg` letters, `ac` shadow/outline, `bg` background, `b2` pattern: FFFFFF 000000 1A0033 3B2B2E F7EEDD FF9CC8 FF2FA0 FF4A55 FF9A3C FFE14D 8FE36B 7DFFC8 5AA8FF 2F3A9E A06BFF C9B5FF
- `t` the words to show. They must appear in your sentence.

In [ ]:
# Your examples (about 90 to start, written like you talk). Add more at the bottom!
MY_EXAMPLES = [
    # --- her own three ---
    ("me dancing and roz bouncing next to me", "av=dance1;fx=bounce;t=roz"),
    ("hi!! in cute pink and sparlkes", "fg=FF9CC8;bp=sparkles;st=outline;t=hi!!"),
    ("me with curly hair gving a kiss face", "av=ooh;hr=curly;t="),

    # --- mini me faces + hair ---
    ("me smiling w straight hair", "av=smile;hr=straight;t="),
    ("just me smiling", "av=smile;t="),
    ("me with my bunny teeth out lol", "av=teeth;t="),
    ("lil me showing my teeth", "av=teeth;t="),
    ("me making a kissy face", "av=ooh;t="),
    ("me blowing kisses with straight hair", "av=ooh;hr=straight;t="),
    ("me going ooh", "av=ooh;t="),
    ("me grinning so big", "av=grin;t="),
    ("me being so happy", "av=grin;t="),
    ("me just chillin", "av=chill;t="),
    ("me lookin unbothered", "av=chill;t="),
    ("chill me with curly hair", "av=chill;hr=curly;t="),

    # --- mini me dances ---
    ("me dancing", "av=dance1;t="),
    ("me dancing w my curls", "av=dance1;hr=curly;t="),
    ("me doing dance 2", "av=dance2;t="),
    ("me doing the second dance with straight hair", "av=dance2;hr=straight;t="),
    ("me doing dance 3 fast", "av=dance3;sp=3;t="),
    ("me jumping around", "av=dance3;t="),
    ("me dancing slow", "av=dance1;sp=1;t="),
    ("me shuffling", "av=dance2;t="),

    # --- me + words ---
    ("me dancing and roz bouncing", "av=dance1;fx=bounce;t=roz"),
    ("me dancing next to ROZ", "av=dance1;t=ROZ"),
    ("ROZ with me dancing", "av=dance1;t=ROZ"),
    ("me grinning and PLUR", "av=grin;t=PLUR"),
    ("me saying hi", "av=smile;t=hi"),
    ("me saying HI!! with my teeth out", "av=teeth;t=HI!!"),
    ("me blowing a kiss and xoxo", "av=ooh;t=xoxo"),
    ("me doing dance 2 and FISHER in green", "av=dance2;fg=8FE36B;t=FISHER"),
    ("kiss face me with LOVE U", "av=ooh;t=LOVE U"),
    ("me chillin and good vibes", "av=chill;t=good vibes"),

    # --- names (fixes "show my name ALEXIA") ---
    ("show my name ALEXIA", "t=ALEXIA"),
    ("my name ALEXIA in pink", "fg=FF9CC8;t=ALEXIA"),
    ("put my name alexia", "t=alexia"),
    ("write my name LEXI", "t=LEXI"),
    ("my name is ALEXIA", "t=ALEXIA"),
    ("my bestie's name MAYA", "t=MAYA"),

    # --- ROZ ---
    ("roz", "t=roz"),
    ("ROZ invert colors fast", "fx=invert;sp=3;t=ROZ"),
    ("roz flashing", "fx=invert;t=roz"),
    ("roz bouncing", "fx=bounce;t=roz"),
    ("roz but make it cute", "fx=bounce;st=outline;fg=FFFFFF;ac=FF2FA0;bg=FF9CC8;bp=hearts;t=roz"),
    ("roz glitchy and green", "fx=glitch;fg=8FE36B;bg=000000;t=roz"),
    ("ROZ in yellow on black flashing", "fx=invert;fg=FFE14D;bg=000000;t=ROZ"),
    ("i love roz with hearts", "bp=hearts;t=i love roz"),
    ("I LOVE ROZ in hot pink bubble letters", "fg=FF2FA0;st=outline;t=I LOVE ROZ"),
    ("roz roz roz rainbow", "fx=rainbow;t=roz roz roz"),

    # --- cute / pink / sparkles ---
    ("hi!! in cute pink", "fg=FF9CC8;st=outline;t=hi!!"),
    ("hii in pink with sparkles", "fg=FF9CC8;bp=sparkles;t=hii"),
    ("plur in cute bubble letters", "st=outline;t=plur"),
    ("xoxo with hearts everywhere", "bp=hearts;t=xoxo"),
    ("kandi kid in pastel pink", "fg=FF9CC8;t=kandi kid"),
    ("cute pink hearts bg", "bp=hearts;b2=FF9CC8;t="),
    ("sparkly purple bg", "bp=sparkles;bg=A06BFF;t="),
    ("sparlkes on black", "bp=sparkles;bg=000000;t="),
    ("lilac sparkles", "bp=sparkles;b2=C9B5FF;t="),
    ("bestie in lavender with stars", "fg=C9B5FF;bp=stars;t=bestie"),

    # --- effects with typos + slang ---
    ("TECHNO glitchy fr", "fx=glitch;t=TECHNO"),
    ("bass wavy and slow", "fx=wave;sp=1;t=bass"),
    ("drop it bouncin", "fx=bounce;t=drop it"),
    ("rave in rainbow letters", "fx=rainbow;t=rave"),
    ("hydrate pulsing", "fx=pulse;t=hydrate"),
    ("free hugs typewriter", "fx=type;t=free hugs"),
    ("lets gooo scrolling fast", "fx=scroll;sp=3;t=lets gooo"),
    ("SLAY flashing super fast", "fx=invert;sp=3;t=SLAY"),
    ("omg wiggly", "fx=wave;t=omg"),
    ("wubs glitch fast green", "fx=glitch;sp=3;fg=8FE36B;t=wubs"),
    ("dom dolla scrollin", "fx=scroll;t=dom dolla"),
    ("sara landry flashing on black", "fx=invert;bg=000000;t=sara landry"),
    ("john summit in orange on navy", "fg=FF9A3C;bg=2F3A9E;t=john summit"),

    # --- backgrounds + colors ---
    ("PLUR on a checkered bg", "bp=checker;t=PLUR"),
    ("peace with stripes", "bp=stripes;t=peace"),
    ("love on polka dots", "bp=dots;t=love"),
    ("vibes with a rainbow background", "bp=rainbow;t=vibes"),
    ("starry night sky with HI", "bp=stars;t=HI"),
    ("pink to purple gradient", "bp=gradient;bg=FF9CC8;b2=A06BFF;t="),
    ("lavender gradient", "bp=gradient;bg=C9B5FF;t="),
    ("baby blue bg with white text BASS", "bg=5AA8FF;fg=FFFFFF;t=BASS"),
    ("trance in mint on black", "fg=7DFFC8;bg=000000;t=trance"),

    # --- vibes ---
    ("spooky vibes BOO", "fx=glitch;fg=FF9A3C;ac=000000;bg=000000;b2=A06BFF;bp=stars;t=BOO"),
    ("cute PLUR", "fx=bounce;st=outline;fg=FFFFFF;ac=FF2FA0;bg=FF9CC8;b2=FFFFFF;bp=hearts;t=PLUR"),
    ("galaxy vibes with me chilling", "fx=pulse;fg=C9B5FF;ac=A06BFF;bg=1A0033;b2=FFFFFF;bp=stars;av=chill;t="),
    ("techno vibes HEADBANG", "fx=glitch;st=plain;fg=7DFFC8;ac=FF2FA0;bg=000000;b2=2F3A9E;bp=stripes;t=HEADBANG"),

    # --- everything at once ---
    ("me dancing and roz bouncing next to me in pink with sparkles", "av=dance1;fx=bounce;fg=FF9CC8;bp=sparkles;t=roz"),
    ("me with curly hair giving a kiss face and xoxo on hearts", "av=ooh;hr=curly;bp=hearts;t=xoxo"),
    ("hi!! in cute pink and sparkles with me smiling", "fg=FF9CC8;bp=sparkles;st=outline;av=smile;t=hi!!"),
    ("me doing dance 3 and TECHNO glitchy on black", "av=dance3;fx=glitch;bg=000000;t=TECHNO"),
    ("roz flashing fast and me grinning", "fx=invert;sp=3;av=grin;t=roz"),
]

## 4. Make the practice examples

In [ ]:
import random, json, time
from scene_data import make_dataset, scene_line
from train import example_from_line
train_ex = make_dataset(40000, seed=1)
val_ex = make_dataset(2000, seed=2)
mine = [example_from_line(p, l) for p, l in MY_EXAMPLES]
train_ex += mine * 30
random.Random(0).shuffle(train_ex)
print(len(train_ex), "practice examples. A few of them:\n")
for e in train_ex[:8]:
    print(" ".join(e["words"]))
    print("    ->", scene_line(e["scene"]), "\n")

## 5. Turn every sentence into meaning numbers with MiniLM
This is the slow part, a few minutes.

In [ ]:
from scene_model import WordVectors
from train import encode
wv = WordVectors("minilm/onnx/model_quantized.onnx", "minilm/tokenizer.json")
t0 = time.time()
train_data = encode(train_ex, wv)
val_data = encode(val_ex, wv)
print(f"done in {time.time() - t0:.0f} s")

## 6. Train!
Each round goes through all the examples once. Watch the scores go up.

In [ ]:
from train import train
model = train(train_data, val_data, epochs=12)

## 7. The real test: sentences it has never seen
These were written by hand, phrased differently from the practice examples.

In [ ]:
from train import check_tests, predict
tests = json.load(open("test_prompts.json"))
score = check_tests(model, wv, tests)

### Try your own sentences

In [ ]:
for s in ["ROZ invert colors fast", "PLUR in pink bubble letters with hearts", "me doing dance 2 with straight hair saying HI"]:
    print(s, "\n   ->", predict(model, wv, s))

## 8. Save it for the app
This saves `scene-ai.bin`. Click the link to download it, then follow the guide to put it in the app.

In [ ]:
from scene_model import export, load_int8, SceneAI
size = export(model, "scene-ai.bin", info=dict(examples=len(train_ex), epochs=12, my_examples=len(MY_EXAMPLES), test_score=round(score, 3)))
print(f"scene-ai.bin: {size / 1024:.0f} KB")
check = load_int8(SceneAI(), "scene-ai.bin")
print("int8 check:"); check_tests(check, wv, tests, log=lambda s: None if s[:1] in "✅❌" else print(s))
from IPython.display import FileLink
FileLink("scene-ai.bin")